# P113 — Aprendizaje por refuerzo profundo que importa

## 1. Título y paper

**Paper:** *Deep Reinforcement Learning That Matters*  
**Autoría:** Peter Henderson, Riashat Islam, Philip Bachman, Joelle Pineau, Doina Precup, David Meger  
**Año y venue:** 2018 · AAAI 2018  
**Nivel:** L3 · **Motor:** `trazabilidad`  
**Ficha completa:** [`P113_trazabilidad`](../../papers/foundational/P113_trazabilidad/README.md)

**Hito:** Demuestra empíricamente que con pocas semillas el ranking entre algoritmos es una moneda al aire, y que muchas mejoras publicadas no sobreviven a la comprobación.

- [doi:10.1609/aaai.v32i1.11694](https://doi.org/10.1609/aaai.v32i1.11694)

> Este notebook implementa una **miniatura** del mecanismo. No reproduce el experimento original ni sus métricas: reproduce la idea para que se pueda inspeccionar y discutir.


## 2. Objetivos

1. Explicar qué problema resolvió el paper: Los resultados en aprendizaje por refuerzo se comparaban con tres o cinco corridas, sin declarar semillas, implementación ni hiperparámetros. Con la varianza real entre semillas, ese protocolo no distingue algoritmos: produce rankings que se invierten al repetir el experimento.
2. Ejecutar una implementación mínima de la propuesta: Medirlo. Ejecutar los mismos algoritmos con muchas semillas, con distintas implementaciones y en distintos entornos, y cuantificar cuánto de la diferencia publicada es señal y cuánto es elección de semilla, de código o de entorno.
3. Predecir el resultado antes de ejecutar, y contrastar la predicción con la salida.
4. Identificar al menos una limitación de la miniatura y una del paper original.
5. Conectar el hito con el siguiente eslabón de la ruta.


## 3. Prerrequisitos

- Python 3.11+ y el paquete del programa instalado (`pip install -e .`).
- Haber leído la guía [método de lectura en 5 pasadas](../../papers/guides/METODO_DE_LECTURA_EN_5_PASADAS.md).
- Hitos previos:
- P63
- P60
- P102


## 4. Intuición

Dos algoritmos con exactamente el mismo rendimiento. Comparados con tres semillas cada uno, uno parece claramente mejor. Repite el experimento y puede salir al revés. Eso no es un problema de los algoritmos: es un problema del protocolo.


## 5. Concepto mínimo

```text
Con k semillas por algoritmo, la media observada tiene desviación σ/√k

    k = 3   →  el ruido tiene tamaño de hallazgo
    k = 30  →  el ruido se encoge y deja de parecerlo

El signo de la diferencia sigue siendo una moneda si los métodos son iguales.
```


## 6. Código explicado

El motor aísla el mecanismo del paper con datos de juguete y salida inspeccionable.


In [ ]:
import json
import pathlib
import sys

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from ai_evolution.papers_lab import run_paper_lab


def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
r = run_paper_lab('trazabilidad', seed=7)['result']
show(r)

## 7. Predicción antes de ejecutar

1. Con 3 semillas, ¿cuántas veces de 200 sale A por debajo de B?
2. ¿Baja esa proporción con 30 semillas?
3. ¿Qué sí baja?

> Escribe tu respuesta aquí antes de continuar.


## 8. Experimento controlado

Se varía una sola cosa y se observa el efecto.


In [ ]:
for semilla in (1, 7, 42):
    r = run_paper_lab('trazabilidad', seed=semilla)
    print(f'semilla {semilla:>2} · evidencia principal:')
    for e in r['evidence']:
        print('   +', e)
    break  # determinista: basta una para ver la estructura
for semilla in (1, 7, 42):
    r = run_paper_lab('trazabilidad', seed=semilla)['result']
    print(f'semilla {semilla:>2} → claves: {list(r)[:4]}')

## 9. Salida interpretable

Con 3 semillas, A sale por debajo en **99 de 200** comparaciones y por encima en 101: una moneda al aire, porque los dos son idénticos. Con 30 semillas la proporción sigue rondando el 0,5. Lo que **sí** baja es la magnitud: la diferencia media observada pasa de **42,0** a **12,2**, y la máxima de **131,5** a **40,6**.


## 10. Comentario pedagógico

Esa es la distinción que hay que tener clara. Con métodos iguales, el signo siempre es aleatorio — y eso está bien. El problema con pocas semillas es que la **magnitud** del ruido tiene tamaño de hallazgo: 131 puntos de diferencia parecen un descubrimiento y son azar.


## 11. Error o anti-patrón deliberado

Anti-patrón: reportar la mejor corrida de un método y la típica del otro.


In [ ]:
print('Con 5 semillas por metodo, reportar el maximo de A y el minimo de B')
print('produce una «mejora» enorme sin que haya ninguna diferencia real.')
print('No hace falta mala fe: basta un formato que no obligue a declarar las corridas.')

## 12. Corrección

El reporte mínimo que hace comparable un resultado:


In [ ]:
r = run_paper_lab('trazabilidad', seed=7)['result']
for e in r['efecto_del_numero_de_semillas']:
    print(f"  {e['semillas_por_algoritmo']:>2} semillas  inversiones={e['proporcion']}"
          f"  dif. media={e['diferencia_media_observada']:<6} dif. maxima={e['diferencia_maxima_observada']}")
print()
print('optimista:', r['reporte_optimista'])
print('honesto  :', r['reporte_honesto'])

## 13. Desafío guiado

Explica por qué la proporción de inversiones NO baja con más semillas, y por qué eso no contradice la utilidad de usar más.


In [ ]:
r = run_paper_lab('trazabilidad', seed=3)['result']
show(r)

## 14. Desafío autónomo

Coge un experimento tuyo, ejecútalo con diez semillas y publica media, desviación, rango y número de corridas. Comprueba si alguna conclusión previa sobrevive.


## 15. Evidencia de aprendizaje

Guarda la tabla del efecto del número de semillas y tu reporte con los tres campos mínimos.

Autoevaluación y respuestas esperadas: [ficha del paper](../../papers/foundational/P113_trazabilidad/README.md) · evaluación formal: [`assessments/papers/P113_trazabilidad.md`](../../assessments/papers/P113_trazabilidad.md)


## 16. Cierre

Los experimentos ya son comparables. Falta que el modelo que sale de ellos venga documentado para quien lo va a usar.


## 17. Conexión con el siguiente hito



Ruta completa: [`papers/ROADMAP.md`](../../papers/ROADMAP.md)
